# Pipeline evaluation — one notebook for the whole iteration cycle

Use this every time you change `build_benefits.py`, prompts, or section codes.
It runs the full pipeline against your API and gives you:

1. **Run** — POST a payload to the API and wait for results
2. **Coverage** — which benefits extracted vs missed (filter mismatch vs LLM failure)
3. **Quality** — diff output rows against a ground-truth reference plan (when available)
4. **Regression scoreboard** — track improvements over time across runs

**Workflow:** Update code → restart Flask → run this notebook → read the bottom-line verdict → iterate.


## 0. Config — set paths and the API URL

In [ ]:
import math
import json
import time
import re
import os
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
import requests
import pandas as pd

# ── Where to test ─────────────────────────────────────────────────────────────
# Use 127.0.0.1 for local Flask; swap to the App Service URL for cloud testing
BASE_URL = 'http://127.0.0.1:8080'

# ── Input + reference paths ───────────────────────────────────────────────────
PAYLOAD_PATH   = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')

# Ground-truth reference. If you don't have one, set REFERENCE_PATH = None and
# the quality-diff section will be skipped.
# Reference file format: same JSON shape as API output (a list of benefit row dicts)
REFERENCE_PATH = None  # e.g. Path('./reference_plans/MOM_H0028_014_0_reference.json')

# ── Scoreboard log ────────────────────────────────────────────────────────────
# Each run appends one row to this CSV so you can track regressions over time
SCOREBOARD_PATH = Path('./eval_scoreboard.csv')

# ── Run config ────────────────────────────────────────────────────────────────
POLL_INTERVAL  = 10
MAX_WAIT       = 900
POST_TIMEOUT   = 120

# ── Optional run label (shows up in scoreboard) ───────────────────────────────
RUN_LABEL = 'v4-HCSC-targets'  # bump this when you change code

print(f'API: {BASE_URL}')
print(f'Payload: {PAYLOAD_PATH.name}')
print(f'Reference: {REFERENCE_PATH.name if REFERENCE_PATH else "none (quality diff will be skipped)"}')
print(f'Run label: {RUN_LABEL}')


## 1. Run the pipeline against the API

POSTs the payload, polls until done. Captures wall-clock time and result count.

In [ ]:
def sanitize(obj):
    if isinstance(obj, dict):  return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)): return None
    return obj

# Health check
r = requests.get(f'{BASE_URL}/', timeout=15)
print(f'Health: {r.status_code}')

# Load + post
raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
rows = raw['pbp'] if isinstance(raw, dict) and 'pbp' in raw else raw
clean = sanitize(rows)

# Distinct plans in input
input_plans = {r['FileName'].strip() for r in clean if r.get('FileName')}
print(f'Input: {len(clean):,} PBP rows / {len(input_plans)} distinct plans\n')

t_post = time.monotonic()
r = requests.post(f'{BASE_URL}/save', json=clean,
                  headers={'Content-Type': 'application/json'}, timeout=POST_TIMEOUT)
post_time = time.monotonic() - t_post

if r.status_code != 202:
    raise RuntimeError(f'POST failed: {r.status_code} {r.text}')

resp = r.json()
load_id = resp['load_id']
print(f'POST: {r.status_code} in {post_time:.1f}s — load_id={load_id}\n')

# Poll until done
elapsed = 0
t_proc = time.monotonic()
output_data = None
while elapsed < MAX_WAIT:
    r = requests.get(f'{BASE_URL}/results/{load_id}', timeout=30)
    data = r.json()
    status = data.get('status')
    if elapsed % 30 == 0 or status != 'processing':
        print(f'  [{elapsed:>4}s] status={status}')
    if status == 'success':
        output_data = data
        break
    if status == 'error':
        raise RuntimeError(f'Pipeline error: {data.get("error")}')
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    raise TimeoutError(f'Timed out after {MAX_WAIT}s')

proc_time = time.monotonic() - t_proc
total_time = post_time + proc_time

output_rows = output_data['results']
print(f'\n✓ Done in {proc_time:.1f}s')
print(f'   Plans processed: {output_data.get("plan_count")}')
print(f'   Total rows out:  {output_data.get("result_count")}')
print(f'   Throughput:      {len(clean)/total_time:.0f} input rows/sec')


## 2. Coverage analysis — what extracted vs what's missing

Per benefit ID across all plans. Distinguishes:
- **OK** — produced output rows in at least some plans
- **DATA_SHAPE** — filter found no matching input rows (BENEFIT_TARGETS issue)
- **LLM_FAILED** — filter found rows, but LLM returned zero (prompt issue)

In [ ]:
# Mirror BENEFIT_TARGETS from build_benefits.py — keep these in sync
BENEFIT_TARGETS = [
    ('610',  'Health Plan Deductible',     [],             ['Health Plan Deductible', 'Medical Deductible', 'Annual Plan Deductible']),
    ('611',  'Drug Deductible',            [],             ['Drug Deductible', 'Rx Deductible', 'Enter Deductible Amount']),
    ('615',  'Drug Monthly Premium',       [],             ['Drug Monthly Premium', 'Rx Premium', 'Part D Premium']),
    ('616',  'Part B Premium Reduction',   [],             ['Part B Premium', 'Part B Reduction', 'Part B giveback']),
    ('620',  'Out-of-Pocket Spending',     [],             ['MOOP', 'Max Enrollee Cost', 'Out of Pocket', 'OOP']),
    ('700',  'Tier Names',                 [],             ['Rx Tier', 'Formulary Tier', 'Tier Names']),
    ('710',  'Initial Coverage',           [],             ['Initial Coverage Phase', 'Rx Setup']),
    ('711',  'Retail Pharmacy',            [],             ['Retail Pharmacy', 'Initial Coverage Phase']),
    ('730',  'Catastrophic Coverage',      [],             ['Catastrophic Coverage']),
    ('740',  'Formulary Exception',        [],             ['Formulary Exception']),
    ('755',  'Preferred Mail Order',       [],             ['Preferred Mail Order']),
    ('760',  'Standard Mail Order',        [],             ['Standard Mail Order']),
    ('800',  'Inpatient Hospital',         ['1a'],         ['Inpatient Hospital']),
    ('810',  'Inpatient Mental Health',    ['1b'],         ['Inpatient Mental Health', 'Inpatient Psychiatric']),
    ('820',  'Skilled Nursing Facility',   ['2'],          ['Skilled Nursing', 'SNF']),
    ('900',  'PCP',                        ['7a'],         ['Primary Care', 'PCP']),
    ('910',  'Specialist',                 ['7b', '7d'],   ['Specialist', 'Specialty Care', 'Physician Specialist']),
    ('911',  'Telehealth',                 ['7d', '7j'],   ['Telehealth', 'Telemedicine', 'Virtual', 'Additional Telehealth', 'Remote Access']),
    ('920',  'Chiropractic',               ['8'],          ['Chiropractic']),
    ('930',  'Podiatry',                   ['9a'],         ['Podiatry']),
    ('940',  'Outpatient Mental Health',   ['4a'],         ['Outpatient Mental Health']),
    ('950',  'Outpatient Substance Abuse', ['4b'],         ['Substance Abuse']),
    ('960',  'Outpatient Surgery',         ['4c','9a1','9a2','9b','9d'], ['Outpatient Surgery', 'Outpatient Hospital', 'Ambulatory Surgical', 'Observation']),
    ('970',  'Ambulance',                  ['5','5a','5b'],['Ambulance']),
    ('981',  'Emergency',                  ['4a','6a'],    ['Emergency Services', 'Emergency Care']),
    ('982',  'Urgent Care',                ['4b','6b'],    ['Urgent Care', 'Urgently Needed']),
    ('990',  'Outpatient Rehab',           ['10','5b'],    ['Outpatient Rehabilitation', 'Physical Therapy', 'Occupational Therapy', 'Cardiac Rehab']),
    ('1000', 'DME',                        ['11a'],        ['Durable Medical Equipment', 'DME']),
    ('1020', 'Diabetes',                   ['11c'],        ['Diabetic', 'Diabetes']),
    ('1030', 'Diagnostic/Lab/Radiology',   ['3','8a2','8b2'], ['Diagnostic', 'X-Ray', 'Lab Services', 'Radiology']),
    ('1050', 'Fitness',                    ['13b','14c4'], ['Fitness', 'SilverSneakers']),
    ('1060', 'Meals',                      ['13c'],        ['Meals']),
    ('1200', 'Kidney',                     ['12'],         ['Kidney', 'Renal', 'Dialysis']),
    ('1300', 'Preventive Dental',          ['16a'],        ['Preventive Dental', 'Dental Services', 'Medicare Dental']),
    ('1301', 'Comprehensive Dental',       ['16b','16c1','16c'], ['Comprehensive Dental', 'Restorative', 'Non-routine Dental']),
    ('1400', 'Hearing - Exams',            ['18a'],        ['Hearing Exam']),
    ('1400', 'Hearing - Aids',             ['18b','18b1'], ['Hearing Aid', 'Prescription Hearing Aid', 'Fitting/Evaluation']),
    ('1500', 'Vision - Eye Exams',         ['17a','17a1'], ['Eye Exam', 'Routine Eye']),
    ('1500', 'Vision - Eyewear',           ['17b','17b2','17b3'], ['Eyewear', 'Eyeglasses', 'Contact Lenses']),
    ('1610', 'Prosthetics',                ['11b'],        ['Prosthetic']),
    ('1700', 'Preventive',                 ['14a','14b','14c'], ['Preventive', 'Wellness Visit', 'Annual Physical']),
    ('1800', 'Transportation',             ['15'],         ['Transportation']),
    ('1900', 'Acupuncture',                ['13a'],        ['Acupuncture', 'Alternative Therapies']),
    ('2100', 'OTC',                        ['13b','13e'],  ['Over-the-Counter', 'OTC']),
]

def row_matches_target(row, codes, keywords):
    cat = (row.get('category') or '').lower()
    for c in codes:
        if f'({c})' in cat or f'({c}1)' in cat or f'({c}2)' in cat:
            return True
    for kw in keywords:
        if kw.lower() in cat:
            return True
    return False

out_df = pd.DataFrame(output_rows)
out_df['benefitid'] = out_df['benefitid'].astype(str)
plans_with_bid = out_df.groupby('benefitid')['planid'].nunique().to_dict()
n_plans = out_df['planid'].nunique()

# Per-benefit verdict
coverage = []
seen_bids = set()
for bid, name, codes, keywords in BENEFIT_TARGETS:
    if bid in seen_bids:
        continue  # split benefits like 1400/1500 — count once
    seen_bids.add(bid)

    plans_with_data = int(plans_with_bid.get(bid, 0))
    if plans_with_data > 0:
        verdict = 'OK'
    else:
        # filter check
        matched = sum(1 for r in clean if row_matches_target(r, codes, keywords))
        verdict = 'DATA_SHAPE' if matched == 0 else 'LLM_FAILED'

    coverage.append({
        'benefitid':       bid,
        'name':            name,
        'plans_w_data':    plans_with_data,
        'pct_plans':       f'{100*plans_with_data/n_plans:.0f}%',
        'verdict':         verdict,
    })

cov_df = pd.DataFrame(coverage)

# Headline counts
n_ok        = (cov_df['verdict']=='OK').sum()
n_dataShape = (cov_df['verdict']=='DATA_SHAPE').sum()
n_llmFailed = (cov_df['verdict']=='LLM_FAILED').sum()
n_total     = len(cov_df)

print(f'Coverage: {n_ok}/{n_total} OK | {n_dataShape} DATA_SHAPE | {n_llmFailed} LLM_FAILED\n')

display(cov_df.sort_values(['verdict', 'plans_w_data'], ascending=[True, False]))


## 3. Quality diff — compare against a ground-truth reference plan

Only runs if `REFERENCE_PATH` is set. Finds the matching planid in the output, diffs row-by-row against the reference, and reports:
- Rows that match exactly
- Rows where benefitdesc differs (text divergence)
- Rows only in reference (LLM missed them)
- Rows only in output (LLM hallucinated or wrong serviceTypeID)

In [ ]:
quality_summary = None

if REFERENCE_PATH and REFERENCE_PATH.exists():
    ref_rows = json.loads(REFERENCE_PATH.read_text(encoding='utf-8'))
    if isinstance(ref_rows, dict) and 'results' in ref_rows:
        ref_rows = ref_rows['results']

    # Find the matching planid in current output
    ref_planid = ref_rows[0].get('planid')
    matched_output = [r for r in output_rows if r.get('planid') == ref_planid]

    print(f'Reference plan: {ref_planid}')
    print(f'  Reference rows: {len(ref_rows)}')
    print(f'  Output rows:    {len(matched_output)}\n')

    if not matched_output:
        print(f'⚠ No output rows for planid {ref_planid} — skipping diff')
    else:
        # Index both by (benefitid, serviceTypeID)
        def key(r): return (str(r.get('benefitid','')), str(r.get('serviceTypeID','')))
        ref_idx = {key(r): r for r in ref_rows}
        out_idx = {key(r): r for r in matched_output}

        keys_ref = set(ref_idx)
        keys_out = set(out_idx)

        exact_match    = []
        desc_mismatch  = []
        missing_in_out = []
        extra_in_out   = []

        for k in keys_ref & keys_out:
            r_ref = ref_idx[k]
            r_out = out_idx[k]
            if r_ref.get('benefitdesc') == r_out.get('benefitdesc') and \
               r_ref.get('tinyDescription') == r_out.get('tinyDescription'):
                exact_match.append(k)
            else:
                desc_mismatch.append({
                    'key':           k,
                    'benefitname':   r_ref.get('benefitname'),
                    'ref_desc':      r_ref.get('benefitdesc'),
                    'out_desc':      r_out.get('benefitdesc'),
                    'ref_tiny':      r_ref.get('tinyDescription'),
                    'out_tiny':      r_out.get('tinyDescription'),
                })

        for k in keys_ref - keys_out:
            r = ref_idx[k]
            missing_in_out.append({
                'key':         k,
                'benefitname': r.get('benefitname'),
                'desc':        r.get('benefitdesc'),
            })

        for k in keys_out - keys_ref:
            r = out_idx[k]
            extra_in_out.append({
                'key':         k,
                'benefitname': r.get('benefitname'),
                'desc':        r.get('benefitdesc'),
            })

        n_total = len(keys_ref | keys_out)
        accuracy = 100 * len(exact_match) / max(n_total, 1)

        print(f'Exact match:         {len(exact_match):>4} rows')
        print(f'Description differs: {len(desc_mismatch):>4} rows')
        print(f'Missing in output:   {len(missing_in_out):>4} rows  (LLM failed to produce)')
        print(f'Extra in output:     {len(extra_in_out):>4} rows  (LLM hallucinated or wrong key)')
        print(f'\nAccuracy: {accuracy:.1f}% exact ({len(exact_match)}/{n_total})')

        quality_summary = {
            'exact':    len(exact_match),
            'differs':  len(desc_mismatch),
            'missing':  len(missing_in_out),
            'extra':    len(extra_in_out),
            'accuracy': accuracy,
        }

        if desc_mismatch:
            print(f'\nTop 10 description mismatches:')
            display(pd.DataFrame(desc_mismatch).head(10))

        if missing_in_out:
            print(f'\nTop 10 missing rows (LLM didn\'t produce these):')
            display(pd.DataFrame(missing_in_out).head(10))

        if extra_in_out:
            print(f'\nTop 10 extra rows (in output but not reference):')
            display(pd.DataFrame(extra_in_out).head(10))
else:
    print('No REFERENCE_PATH set — skipping quality diff.')
    print('To enable, save a known-good output as JSON and set REFERENCE_PATH in cell 0.')


## 4. Bottom-line verdict — what to fix next

Tells you the single most impactful thing to change before the next iteration.

In [ ]:
print('=' * 70)
print(f'Run: {RUN_LABEL}    Total time: {total_time:.1f}s')
print('=' * 70)
print(f'Plans: {output_data.get("plan_count")} processed | {output_data.get("result_count")} rows out')
print(f'Coverage: {n_ok}/{n_total} benefits OK | {n_dataShape} data-shape gaps | {n_llmFailed} LLM failures')

if quality_summary:
    print(f'Quality: {quality_summary["accuracy"]:.1f}% exact match against reference')

print()
print('NEXT ACTION:')
if n_dataShape > n_llmFailed and n_dataShape > 2:
    failing = cov_df[cov_df['verdict']=='DATA_SHAPE']['name'].tolist()
    print(f'  → Update BENEFIT_TARGETS for: {", ".join(failing)}')
    print(f'    Look at category strings in input data, add matching keywords/codes')
elif n_llmFailed > 0:
    failing = cov_df[cov_df['verdict']=='LLM_FAILED']['name'].tolist()
    print(f'  → LLM is bailing on: {", ".join(failing)}')
    print(f'    Either split these into sub-targets (like Vision/Hearing) or')
    print(f'    add a focused few-shot example for the specific failure mode')
elif quality_summary and quality_summary['accuracy'] < 80:
    print(f'  → Coverage looks good but quality is {quality_summary["accuracy"]:.1f}%.')
    print(f'    Inspect the description mismatches above — likely a formatting-rule')
    print(f'    or few-shot prompt issue. Pick the top 3 patterns and add examples.')
elif quality_summary and quality_summary['accuracy'] >= 95:
    print(f'  ✓ Quality is at {quality_summary["accuracy"]:.1f}%. Ship it.')
else:
    print(f'  ✓ All benefits extracting. No reference to validate quality against.')
    print(f'    Save this run\'s output as a reference for future regression checks.')


## 5. Save to scoreboard (regression tracking)

Appends this run's stats to a CSV. Over time you can see whether each iteration is helping or hurting.

In [ ]:
scoreboard_row = {
    'timestamp':      datetime.now().isoformat(timespec='seconds'),
    'run_label':      RUN_LABEL,
    'load_id':        load_id,
    'input_plans':    len(input_plans),
    'output_plans':   output_data.get('plan_count'),
    'output_rows':    output_data.get('result_count'),
    'wall_time_s':    round(total_time, 1),
    'rows_per_sec':   round(len(clean)/total_time, 0),
    'benefits_ok':    n_ok,
    'benefits_data_shape': n_dataShape,
    'benefits_llm_failed': n_llmFailed,
    'quality_exact':  quality_summary['exact'] if quality_summary else None,
    'quality_acc_pct': round(quality_summary['accuracy'], 1) if quality_summary else None,
}

sb_df = pd.DataFrame([scoreboard_row])
if SCOREBOARD_PATH.exists():
    existing = pd.read_csv(SCOREBOARD_PATH)
    sb_df = pd.concat([existing, sb_df], ignore_index=True)
sb_df.to_csv(SCOREBOARD_PATH, index=False)

print(f'Appended to {SCOREBOARD_PATH}\n')
print('Recent runs:')
display(sb_df.tail(10))


## 6. Optional — save this output as a reference for next time

Once you have a run that looks correct (manually verified), save it as the new reference. Future runs can be diffed against it to catch regressions.

In [ ]:
# Uncomment to save:
# ref_name = f'reference_{load_id}.json'
# Path(ref_name).write_text(json.dumps(output_rows, indent=2))
# print(f'Saved {len(output_rows)} rows to {ref_name}')
# print('Set REFERENCE_PATH in cell 0 to use this for next iteration.')
print('Reference-save is commented out by default. Uncomment when you have a verified run.')
